<a href="https://colab.research.google.com/github/nikhitarao/nikhitarao_projects/blob/Healthcare-Insurance-Medical-Inflation-Analysis-using-Synthetic-Data/Inpatient_Healthcare_Medical_Procedure_Synthetic_Data_Generation_Policy_Dataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Health Insurance Policy Database for a Fictional New York based Health Insurer

#### Import Libraries

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import os

#### Define the starting constants



In [2]:
# Constants
start_year = 2010
end_year = 2023
base_year_policy_count = 50000
growth_rate = 0.05
fluctuation = 0.02

#### Define the Quarterly Distribution for New Business for seasonality

In [3]:
# Quarterly distribution for New Business
quarter_distribution = {
    "Q1": 0.30,
    "Q2": 0.20,
    "Q3": 0.15,
    "Q4": 0.35
}


#### Yearly Policy Counts Calculation with growth rate and fluctuations

In [4]:
# Function to calculate yearly policy counts with growth and fluctuation
def calculate_policy_count(base_count, years, growth_rate, fluctuation):
    yearly_counts = []
    for year in range(years):
        fluctuation_factor = np.random.uniform(-fluctuation, fluctuation)
        annual_growth = growth_rate + fluctuation_factor
        base_count = int(base_count * (1 + annual_growth))
        yearly_counts.append(base_count)
    return yearly_counts


In [5]:
# Generate yearly policy counts
years = end_year - start_year + 1
yearly_policy_counts = calculate_policy_count(base_year_policy_count, years, growth_rate, fluctuation)

#### Generating random dates with the Quarters

In [6]:
# Function to generate random dates within quarters
def generate_dates(year, quarter, count):
    quarter_ranges = {
        "Q1": (f"{year}-01-01", f"{year}-03-31"),
        "Q2": (f"{year}-04-01", f"{year}-06-30"),
        "Q3": (f"{year}-07-01", f"{year}-09-30"),
        "Q4": (f"{year}-10-01", f"{year}-12-31")
    }
    start_date, end_date = quarter_ranges[quarter]
    start_date = datetime.strptime(start_date, "%Y-%m-%d")
    end_date = datetime.strptime(end_date, "%Y-%m-%d")
    return [start_date + timedelta(days=np.random.randint(0, (end_date - start_date).days + 1)) for _ in range(count)]


#### Data Type Definition for the Dataframe

In [7]:
# Define the data types for the DataFrame
column_dtypes = {
    "Business Type": "object",
    "Policy Start Date": "object",
    "Policy End Date": "object",
    "Churn Flag": "object",
    "Renewal Flag": "object",
    "Policyholder ID": "int64",
    "Location Type": "object",
    "Policy Type": "object",
    "Policy ID": "object",
    "Group Type": "object",
    "Annual Family Income": "float64",
    "Corporation Type": "object",
    "Industry Type": "object",
    "Annual Revenue": "float64",
    "Annual Payroll": "float64",
    "Number of Members": "int64",
    "Coverage Amount": "float64",
    "Policy Annual Premium": "float64",
    "Deductible or CoPay Flag": "object",
    "Deductible Amount": "int64",
    "CoPay Percentage": "float64"
}


#### Dataframe Creation with Business Type, Policy Start Date, Policy End Date populated

In [8]:
# Create an empty DataFrame with specified dtypes
dataset = pd.DataFrame(columns=column_dtypes.keys()).astype(column_dtypes)

In [9]:
# Placeholder for bulk data storage
data = []

policyholder_id_start = 10000000  # Start of 8-digit policyholder IDs
current_policyholder_id = policyholder_id_start  # Initialize the policyholder ID counter

for year, count in zip(range(start_year, end_year + 1), yearly_policy_counts):
    if year == 2010:
        # All policies in 2010 are "New Business"
        start_dates = [datetime(year, 1, 1) + timedelta(days=np.random.randint(0, 365)) for _ in range(count)]
        for start_date in start_dates:
            policy_end_date = start_date + timedelta(days=365 + (1 if start_date.year % 4 == 0 and (start_date.year % 100 != 0 or start_date.year % 400 == 0) else 0))

            data.append([
                "New Business", start_date.strftime("%Y-%m-%d"), policy_end_date.strftime("%Y-%m-%d"), None, None,
                current_policyholder_id, None, None, None, None, None, None, None, None, None, None, None, None,
                None, None, None  # Fill for the 21 columns
            ])
            current_policyholder_id += 1
    else:
        # Split into "New Business" and "Renewal" from 2011 onwards
        new_business_count = int(count * sum(quarter_distribution.values()))
        renewal_count = count - new_business_count

        # Distribute New Business across quarters
        quarterly_counts = {q: int(new_business_count * p) for q, p in quarter_distribution.items()}

        for quarter, quarter_count in quarterly_counts.items():
            start_dates = generate_dates(year, quarter, quarter_count)
            for start_date in start_dates:
                policy_end_date = start_date + timedelta(days=365 + (1 if start_date.year % 4 == 0 and (start_date.year % 100 != 0 or start_date.year % 400 == 0) else 0))

                data.append([
                    "New Business", start_date.strftime("%Y-%m-%d"), policy_end_date.strftime("%Y-%m-%d"), None, None,
                    current_policyholder_id, None, None, None, None, None, None, None, None, None, None, None, None,
                    None, None, None  # Fill for the 21 columns
                ])
                current_policyholder_id += 1

        # Generate Renewal policies
        renewal_dates = [datetime(year, 1, 1) + timedelta(days=np.random.randint(0, 365)) for _ in range(renewal_count)]
        for renewal_date in renewal_dates:
            policy_end_date = renewal_date + timedelta(days=365 + (1 if renewal_date.year % 4 == 0 and (renewal_date.year % 100 != 0 or renewal_date.year % 400 == 0) else 0))

            data.append([
                "Renewal", renewal_date.strftime("%Y-%m-%d"), policy_end_date.strftime("%Y-%m-%d"), None, None,
                current_policyholder_id, None, None, None, None, None, None, None, None, None, None, None, None,
                None, None, None  # Fill for the 21 columns
            ])
            current_policyholder_id += 1

# Create DataFrame in one step
dataset = pd.DataFrame(data, columns=column_dtypes.keys())
print(f"Populated dataset with {len(dataset)} rows.")

dataset.head()

Populated dataset with 1002691 rows.


,Business Type,Policy Start Date,Policy End Date,Churn Flag,Renewal Flag,Policyholder ID,Location Type,Policy Type,Policy ID,Group Type,...,Corporation Type,Industry Type,Annual Revenue,Annual Payroll,Number of Members,Coverage Amount,Policy Annual Premium,Deductible or CoPay Flag,Deductible Amount,CoPay Percentage
0,New Business,2010-03-15,2011-03-15,None,None,10000000,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,New Business,2010-06-26,2011-06-26,None,None,10000001,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,New Business,2010-10-11,2011-10-11,None,None,10000002,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,New Business,2010-10-03,2011-10-03,None,None,10000003,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,New Business,2010-04-22,2011-04-22,None,None,10000004,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [10]:
dataset.shape

(1002691, 21)

#### Adding values for columns Churn Flag, Renewal Flag, Policyholder ID


In [11]:
# Initialize policyholder map and churn rates
policyholder_map = {}  # Tracks Policyholder IDs and their status
churn_rate_range = (0.10, 0.20)  # Churn rate range
renewal_upsell_rate = 0.03  # Minimum upsell rate for renewals

# Add columns for processing
dataset["Churn Flag"] = None
dataset["Renewal Flag"] = None
dataset["Policyholder ID"] = None

# Track the last Policyholder ID
policyholder_id_counter = 1

# Process each year
for year in range(start_year, end_year + 1):
    # Extract data for the current year
    current_year_data = dataset[dataset["Policy Start Date"].str.startswith(str(year))].copy()

    # Separate "New Business" and "Renewal"
    new_business = current_year_data[current_year_data["Business Type"] == "New Business"]
    renewal = current_year_data[current_year_data["Business Type"] == "Renewal"]

    # Assign Policyholder IDs to new business
    new_business["Policyholder ID"] = np.arange(policyholder_id_counter, policyholder_id_counter + len(new_business))
    policyholder_id_counter += len(new_business)

    # Update policyholder map for new business
    for pid in new_business["Policyholder ID"]:
        policyholder_map[pid] = {"status": "Active", "year": year}

    # Process renewals
    if year > start_year:  # Renewals start from the second year
        previous_year = year - 1

        # Identify active IDs from the previous year
        active_ids = [
            pid for pid, details in policyholder_map.items()
            if details["status"] == "Active" and details["year"] == previous_year
        ]

        # Determine churn
        churn_rate = np.random.uniform(*churn_rate_range)
        churn_count = int(len(active_ids) * churn_rate)
        churned_ids = set(np.random.choice(active_ids, churn_count, replace=False))
        renewed_ids = set(active_ids) - churned_ids

        # Update policyholder map for churned and renewed policies
        for pid in churned_ids:
            policyholder_map[pid]["status"] = "Churned"
        for pid in renewed_ids:
            policyholder_map[pid]["year"] = year

        # Assign Renewal and Churn Flags
        renewal["Renewal Flag"] = renewal["Policyholder ID"].apply(
            lambda pid: "Renewed with Up-sell" if pid in renewed_ids and np.random.random() < renewal_upsell_rate else
                        ("Renewed As-is" if pid in renewed_ids else "Not Renewed")
        )
        renewal["Churn Flag"] = renewal["Policyholder ID"].apply(
            lambda pid: "Not Churn" if pid in renewed_ids else "Churn"
        )

    # Combine new business and processed renewals
    processed_data = pd.concat([new_business, renewal], ignore_index=True)

    # Ensure proper index alignment and update dataset
    dataset.loc[processed_data.index, ["Churn Flag", "Renewal Flag", "Policyholder ID"]] = processed_data[
        ["Churn Flag", "Renewal Flag", "Policyholder ID"]
    ]

print("Processing completed!")
dataset.head()

Processing completed!


,Business Type,Policy Start Date,Policy End Date,Churn Flag,Renewal Flag,Policyholder ID,Location Type,Policy Type,Policy ID,Group Type,...,Corporation Type,Industry Type,Annual Revenue,Annual Payroll,Number of Members,Coverage Amount,Policy Annual Premium,Deductible or CoPay Flag,Deductible Amount,CoPay Percentage
0,New Business,2010-03-15,2011-03-15,None,None,906466,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
1,New Business,2010-06-26,2011-06-26,None,None,906467,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
2,New Business,2010-10-11,2011-10-11,None,None,906468,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
3,New Business,2010-10-03,2011-10-03,None,None,906469,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None
4,New Business,2010-04-22,2011-04-22,None,None,906470,None,None,None,None,...,None,None,None,None,None,None,None,None,None,None


In [12]:
dataset.shape

(1002691, 21)

#### Adding values for columns Location Type, Policy Type, Policy ID

In [13]:
# Add columns for processing
dataset["Location Type"] = None
dataset["Policy Type"] = None
dataset["Policy ID"] = None

# Location distribution adjustments
base_distributions = {"Urban": 0.65, "Suburban": 0.25, "Rural": 0.10}
yearly_adjustments = {
    "Urban": lambda year: 0.65 - ((year - 2010) * 0.05 / 13),  # Linear decrease to 60%
    "Suburban": lambda year: 0.25 + ((year - 2010) * 0.05 / 13),  # Linear increase to 30%
    "Rural": lambda year: 0.10  # Stable around 10%
}

# Policy type definitions and distribution rules
policy_types = [
    {"Code": "I", "IntroYear": 2012, "Population": 0.30, "LocationDist": {"Urban": 0.50, "Suburban": 0.30, "Rural": 0.20}},
    {"Code": "F", "IntroYear": 2012, "Population": 0.25, "LocationDist": {"Urban": 0.40, "Suburban": 0.40, "Rural": 0.20}},
    {"Code": "G", "IntroYear": 2010, "Population": 0.30, "LocationDist": {"Urban": 0.60, "Suburban": 0.30, "Rural": 0.10}},
    {"Code": "M", "IntroYear": 2016, "Population": 0.10, "LocationDist": {"Urban": 0.30, "Suburban": 0.40, "Rural": 0.30}},
    {"Code": "MA", "IntroYear": 2014, "Population": 0.05, "LocationDist": {"Urban": 0.25, "Suburban": 0.50, "Rural": 0.25}},
    {"Code": "HD", "IntroYear": 2014, "Population": 0.10, "LocationDist": {"Urban": 0.55, "Suburban": 0.35, "Rural": 0.10}},
    {"Code": "MCP-HMO", "IntroYear": 2015, "Population": 0.25, "LocationDist": {"Urban": 0.50, "Suburban": 0.30, "Rural": 0.20}},
    {"Code": "MCP-PPO", "IntroYear": 2015, "Population": 0.25, "LocationDist": {"Urban": 0.45, "Suburban": 0.35, "Rural": 0.20}},
    {"Code": "SNP", "IntroYear": 2017, "Population": 0.01, "LocationDist": {"Urban": 0.30, "Suburban": 0.50, "Rural": 0.20}},
    {"Code": "SHI", "IntroYear": 2019, "Population": 0.01, "LocationDist": {"Urban": 0.60, "Suburban": 0.30, "Rural": 0.10}},
    {"Code": "CHI", "IntroYear": 2018, "Population": 0.01, "LocationDist": {"Urban": 0.70, "Suburban": 0.20, "Rural": 0.10}},
    {"Code": "DVI", "IntroYear": 2020, "Population": 0.03, "LocationDist": {"Urban": 0.40, "Suburban": 0.40, "Rural": 0.20}}
]

# Generate Policy ID
policy_id_counter = 10000000  # Start of 8-digit Policy IDs

# Process each year
for year in range(start_year, end_year + 1):
    # Calculate location distribution for the current year
    loc_dist = {loc: yearly_adjustments[loc](year) for loc in base_distributions}

    # Extract data for the current year
    current_year_data = dataset[dataset["Policy Start Date"].str.startswith(str(year))].copy()

    # Assign Location Type
    current_year_data["Location Type"] = np.random.choice(
        ["Urban", "Suburban", "Rural"], size=len(current_year_data), p=list(loc_dist.values())
    )

    # Assign Policy Type
    eligible_policy_types = [ptype for ptype in policy_types if ptype["IntroYear"] <= year]
    policy_type_weights = [ptype["Population"] for ptype in eligible_policy_types]

    # Normalize probabilities to sum to 1
    policy_type_weights = np.array(policy_type_weights) / sum(policy_type_weights)

    current_year_data["Policy Type"] = np.random.choice(
        [ptype["Code"] for ptype in eligible_policy_types], size=len(current_year_data), p=policy_type_weights
    )

    # Adjust Policy Type based on Location Distribution
    for loc in loc_dist:
        loc_indices = current_year_data[current_year_data["Location Type"] == loc].index
        loc_eligible_types = [
            ptype for ptype in eligible_policy_types if loc in ptype["LocationDist"]
        ]
        loc_weights = [ptype["LocationDist"][loc] for ptype in loc_eligible_types]

        # Normalize location weights
        loc_weights = np.array(loc_weights) / sum(loc_weights)

        current_year_data.loc[loc_indices, "Policy Type"] = np.random.choice(
            [ptype["Code"] for ptype in loc_eligible_types], size=len(loc_indices), p=loc_weights
        )

    # Assign Policy ID as an 8-digit code
    current_year_data["Policy ID"] = np.arange(policy_id_counter, policy_id_counter + len(current_year_data))
    policy_id_counter += len(current_year_data)

    # Update dataset
    dataset.loc[current_year_data.index, ["Location Type", "Policy Type", "Policy ID"]] = current_year_data[
        ["Location Type", "Policy Type", "Policy ID"]
    ]

print("Location Type, Policy Type, and 8-digit Policy ID assignment completed!")

dataset.head()

Location Type, Policy Type, and 8-digit Policy ID assignment completed!


,Business Type,Policy Start Date,Policy End Date,Churn Flag,Renewal Flag,Policyholder ID,Location Type,Policy Type,Policy ID,Group Type,...,Corporation Type,Industry Type,Annual Revenue,Annual Payroll,Number of Members,Coverage Amount,Policy Annual Premium,Deductible or CoPay Flag,Deductible Amount,CoPay Percentage
0,New Business,2010-03-15,2011-03-15,None,None,906466,Urban,G,10000000,None,...,None,None,None,None,None,None,None,None,None,None
1,New Business,2010-06-26,2011-06-26,None,None,906467,Urban,G,10000001,None,...,None,None,None,None,None,None,None,None,None,None
2,New Business,2010-10-11,2011-10-11,None,None,906468,Urban,G,10000002,None,...,None,None,None,None,None,None,None,None,None,None
3,New Business,2010-10-03,2011-10-03,None,None,906469,Urban,G,10000003,None,...,None,None,None,None,None,None,None,None,None,None
4,New Business,2010-04-22,2011-04-22,None,None,906470,Suburban,G,10000004,None,...,None,None,None,None,None,None,None,None,None,None


In [14]:
dataset.shape

(1002691, 21)

#### Adding values for columns Group Type, Annual Family Income, Corporation Type, Industry Type, Annual Revenue, Annual Payroll

In [15]:
# Add new columns to the dataset
dataset["Group Type"] = None
dataset["Annual Family Income"] = None
dataset["Corporation Type"] = None
dataset["Industry Type"] = None
dataset["Annual Revenue"] = None
dataset["Annual Payroll"] = None

# Define distributions and value ranges
group_type_dist = {
    "Urban": [0.30, 0.70],
    "Suburban": [0.55, 0.45],
    "Rural": [0.75, 0.25],
}

family_income_ranges = {
    "Urban": (75000, 120000),
    "Suburban": (65000, 100000),
    "Rural": (50000, 80000),
}

median_income_growth_rate = 0.03  # 3% growth per year
random_fluctuation = 0.02  # ±2%

# Ensure Year column exists
if "Year" not in dataset.columns:
    dataset["Year"] = pd.to_datetime(dataset["Policy Start Date"]).dt.year

# Vectorized Group Type Assignment
location_types = dataset["Location Type"].values
group_types = np.empty(len(location_types), dtype=object)

for location, probs in group_type_dist.items():
    mask = location_types == location
    group_types[mask] = np.random.choice(["Family", "Group Health"], size=mask.sum(), p=probs)

dataset["Group Type"] = group_types

# Annual Family Income with Growth
family_mask = dataset["Group Type"] == "Family"
for location, income_range in family_income_ranges.items():
    mask = family_mask & (dataset["Location Type"] == location)
    base_income = np.random.randint(income_range[0], income_range[1] + 1, size=mask.sum())
    growth_rate = np.random.uniform(
        median_income_growth_rate - random_fluctuation,
        median_income_growth_rate + random_fluctuation,
        size=mask.sum()
    )
    years_since_base = dataset.loc[mask, "Year"] - start_year  # Assuming `start_year` is defined
    adjusted_income = base_income * ((1 + growth_rate) ** years_since_base)
    dataset.loc[mask, "Annual Family Income"] = adjusted_income.astype(int)

# Corporation Type for Group Health
group_health_mask = dataset["Group Type"] == "Group Health"
corporation_type_dist = {
    "Urban": {"Large Corporation": 0.70, "Medium Corporation": 0.30},
    "Suburban": {"Medium Corporation": 1.00},
    "Rural": {"Small Corporation": 1.00},
}

for location, corp_probs in corporation_type_dist.items():
    mask = group_health_mask & (dataset["Location Type"] == location)
    corp_types = np.random.choice(
        list(corp_probs.keys()), size=mask.sum(), p=list(corp_probs.values())
    )
    dataset.loc[mask, "Corporation Type"] = corp_types

# Industry Type and Financials with Dynamic Adjustments
industry_type_dist = {
    "Urban": {
        "Large Corporation": {"Tech": 0.25, "Finance": 0.45},
        "Medium Corporation": {"Healthcare": 0.10, "Corporate Services": 0.20},
    },
    "Suburban": {
        "Medium Corporation": {"Manufacturing": 0.25, "Retail": 0.35},
        "Small Corporation": {"Education": 0.15, "Local Businesses": 0.25},
    },
    "Rural": {
        "Small Corporation": {"Agriculture": 0.40, "Small Manufacturers": 0.60},
    },
}

revenue_ranges = {
    "Urban": {
        "Tech": (50000000, 100000000),
        "Finance": (50000000, 100000000),
        "Healthcare": (10000000, 50000000),
        "Corporate Services": (10000000, 50000000),
    },
    "Suburban": {
        "Manufacturing": (5000000, 30000000),
        "Retail": (5000000, 30000000),
        "Education": (1000000, 10000000),
        "Local Businesses": (1000000, 5000000),
    },
    "Rural": {
        "Agriculture": (1000000, 5000000),
        "Small Manufacturers": (5000000, 30000000),
    },
}

payroll_ranges = {
    "Urban": {
        "Tech": (10000000, 20000000),
        "Finance": (10000000, 20000000),
        "Healthcare": (5000000, 25000000),
        "Corporate Services": (5000000, 25000000),
    },
    "Suburban": {
        "Manufacturing": (2000000, 15000000),
        "Retail": (2000000, 15000000),
        "Education": (500000, 5000000),
        "Local Businesses": (500000, 5000000),
    },
    "Rural": {
        "Agriculture": (500000, 2000000),
        "Small Manufacturers": (2000000, 15000000),
    },
}

for location, industry_map in industry_type_dist.items():
    corp_types = dataset.loc[group_health_mask & (dataset["Location Type"] == location), "Corporation Type"].unique()
    for corp_type in corp_types:
        industry_probs = industry_map.get(corp_type, {})
        industry_probs_norm = np.array(list(industry_probs.values())) / sum(industry_probs.values())

        mask = (dataset["Corporation Type"] == corp_type) & (dataset["Location Type"] == location)
        industries = np.random.choice(
            list(industry_probs.keys()), size=mask.sum(), p=industry_probs_norm
        )
        dataset.loc[mask, "Industry Type"] = industries

        for industry in list(industry_probs.keys()):
            industry_mask = mask & (dataset["Industry Type"] == industry)
            industry_size = industry_mask.sum()

            # Assign Annual Revenue
            if revenue_range := revenue_ranges[location].get(industry):
                base_revenue = np.random.randint(revenue_range[0], revenue_range[1] + 1, size=industry_size)
                revenue_growth = np.random.uniform(0.03, 0.05, size=industry_size)
                years_since_base = dataset.loc[industry_mask, "Year"] - start_year
                adjusted_revenue = base_revenue * ((1 + revenue_growth) ** years_since_base)
                dataset.loc[industry_mask, "Annual Revenue"] = adjusted_revenue.astype(int)

            # Assign Annual Payroll
            if payroll_range := payroll_ranges[location].get(industry):
                base_payroll = np.random.randint(payroll_range[0], payroll_range[1] + 1, size=industry_size)
                payroll_growth = np.random.uniform(0.03, 0.05, size=industry_size)
                adjusted_payroll = base_payroll * ((1 + payroll_growth) ** years_since_base)
                dataset.loc[industry_mask, "Annual Payroll"] = adjusted_payroll.astype(int)

print("Updated dataset with Group Type, Income, Corporation, Industry, Revenue, and Payroll.")

dataset.head()

Updated dataset with Group Type, Income, Corporation, Industry, Revenue, and Payroll.


,Business Type,Policy Start Date,Policy End Date,Churn Flag,Renewal Flag,Policyholder ID,Location Type,Policy Type,Policy ID,Group Type,...,Industry Type,Annual Revenue,Annual Payroll,Number of Members,Coverage Amount,Policy Annual Premium,Deductible or CoPay Flag,Deductible Amount,CoPay Percentage,Year
0,New Business,2010-03-15,2011-03-15,None,None,906466,Urban,G,10000000,Family,...,None,None,None,None,None,None,None,None,None,2010
1,New Business,2010-06-26,2011-06-26,None,None,906467,Urban,G,10000001,Group Health,...,Finance,55953399,13443077,None,None,None,None,None,None,2010
2,New Business,2010-10-11,2011-10-11,None,None,906468,Urban,G,10000002,Group Health,...,Corporate Services,40825559,22908871,None,None,None,None,None,None,2010
3,New Business,2010-10-03,2011-10-03,None,None,906469,Urban,G,10000003,Group Health,...,Tech,78048354,11453662,None,None,None,None,None,None,2010
4,New Business,2010-04-22,2011-04-22,None,None,906470,Suburban,G,10000004,Family,...,None,None,None,None,None,None,None,None,None,2010


In [16]:
dataset.shape

(1002691, 22)

#### Adding values for columns Number of Members, Coverage Amount and Policy Annual Premium


In [17]:
# Add new columns for processing
dataset["Number of Members"] = None
dataset["Coverage Amount"] = None
dataset["Policy Annual Premium"] = None

# Define member distributions for Group Type
members_dist_family = {
    "Urban": (2, 5),
    "Suburban": (2, 6),
    "Rural": (2, 8),
}

members_dist_group = {
    "Urban": {"Tech and Finance": (200, 1000, 0.7), "Healthcare and Corporate Services": (100, 500, 0.3)},
    "Suburban": {"Manufacturing and Retail": (50, 300, 0.6), "Education": (30, 200, 0.4)},
    "Rural": {"Agriculture and Small Manufacturers": (25, 200, 1.0)},
}

# Coverage amount ranges for Policy Type
coverage_ranges = {
    "I": (10000, 50000),
    "F": (50000, 200000),
    "G": (500000, 5000000),
    "M": (20000, 75000),
    "MA": (50000, 100000),
    "HD": (5000, 20000),
    "SHI": (1000, 10000),
    "DVI": (1000, 5000),
    "SNP": (50000, 75000),
    "CHI": (1000, 10000),
}

# Premium ranges for Policy Type
premium_ranges = {
    "I": (500, 2000),
    "F": (2000, 10000),
    "G": (50000, 500000),
    "M": (1000, 5000),
    "MA": (1000, 5000),
    "HD": (400, 1400),
    "SHI": (500, 1000),
    "DVI": (100, 500),
    "SNP": (2000, 5000),
    "CHI": (500, 2000),
}

# Vectorized processing
location_types = dataset["Location Type"].values
group_types = dataset["Group Type"].values
policy_types = dataset["Policy Type"].values
renewal_flags = dataset["Renewal Flag"].values

# Populate Number of Members
for loc, family_range in members_dist_family.items():
    family_mask = (location_types == loc) & (group_types == "Family")
    group_mask = (location_types == loc) & (group_types == "Group Health")

    # Family Members
    if family_mask.sum() > 0:
        base_family_members = np.random.randint(family_range[0], family_range[1] + 1, family_mask.sum())
        renewal_mask = family_mask & (renewal_flags != "New Business")
        renewal_booleans = np.random.choice([True, False], size=renewal_mask.sum(), p=[0.2, 0.8])
        base_family_members[renewal_booleans] += np.random.randint(1, 3, size=renewal_booleans.sum())
        base_family_members = np.clip(base_family_members, family_range[0], family_range[1])
        dataset.loc[family_mask, "Number of Members"] = base_family_members

    # Group Health Members
    for category, (low, high, weight) in members_dist_group[loc].items():
        category_mask = group_mask & dataset["Industry Type"].str.contains(category, na=False)
        if category_mask.sum() > 0:
            base_group_members = np.random.randint(low, high + 1, category_mask.sum())
            payroll_growth = np.random.uniform(1.03, 1.20, size=category_mask.sum())
            dataset.loc[category_mask, "Number of Members"] = (base_group_members * payroll_growth).astype(int)

# Populate Coverage Amount
for policy, (low, high) in coverage_ranges.items():
    mask = policy_types == policy
    if mask.sum() > 0:
        base_coverage = np.random.randint(low, high + 1, mask.sum())
        dataset.loc[mask, "Coverage Amount"] = base_coverage

# Apply adjustments for renewals and location
renewal_as_is_mask = (renewal_flags == "Renewed As-Is")
renewal_upsell_mask = (renewal_flags == "Renewed with Upsell")

dataset.loc[renewal_as_is_mask, "Coverage Amount"] *= np.random.uniform(1.05, 1.07, size=renewal_as_is_mask.sum())
dataset.loc[renewal_upsell_mask, "Coverage Amount"] *= np.random.uniform(1.10, 1.20, size=renewal_upsell_mask.sum())
dataset.loc[location_types == "Urban", "Coverage Amount"] *= 1.1
dataset.loc[location_types == "Suburban", "Coverage Amount"] *= 1.05

# Populate Policy Annual Premium
for policy, (low, high) in premium_ranges.items():
    mask = policy_types == policy
    if mask.sum() > 0:
        base_premium = np.random.randint(low, high + 1, mask.sum())
        dataset.loc[mask, "Policy Annual Premium"] = base_premium

# Apply adjustments for renewals and location
dataset.loc[renewal_as_is_mask, "Policy Annual Premium"] *= np.random.uniform(1.05, 1.07, size=renewal_as_is_mask.sum())
dataset.loc[renewal_upsell_mask, "Policy Annual Premium"] *= np.random.uniform(1.10, 1.20, size=renewal_upsell_mask.sum())
dataset.loc[location_types == "Urban", "Policy Annual Premium"] *= 1.2
dataset.loc[location_types == "Suburban", "Policy Annual Premium"] *= 1.1

# Ensure no null or None values
dataset["Number of Members"] = dataset["Number of Members"].fillna(2).astype(int)
dataset["Coverage Amount"] = dataset["Coverage Amount"].fillna(10000).astype(int)
dataset["Policy Annual Premium"] = dataset["Policy Annual Premium"].fillna(500).astype(int)

print("Updated Number of Members, Coverage Amount, and Policy Annual Premium without missing values.")
dataset.head()

<ipython-input-17-32dffc3699f1>:105: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataset["Number of Members"] = dataset["Number of Members"].fillna(2).astype(int)
<ipython-input-17-32dffc3699f1>:106: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataset["Coverage Amount"] = dataset["Coverage Amount"].fillna(10000).astype(int)


Updated Number of Members, Coverage Amount, and Policy Annual Premium without missing values.


<ipython-input-17-32dffc3699f1>:107: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataset["Policy Annual Premium"] = dataset["Policy Annual Premium"].fillna(500).astype(int)


,Business Type,Policy Start Date,Policy End Date,Churn Flag,Renewal Flag,Policyholder ID,Location Type,Policy Type,Policy ID,Group Type,...,Industry Type,Annual Revenue,Annual Payroll,Number of Members,Coverage Amount,Policy Annual Premium,Deductible or CoPay Flag,Deductible Amount,CoPay Percentage,Year
0,New Business,2010-03-15,2011-03-15,None,None,906466,Urban,G,10000000,Family,...,None,None,None,3,1856126,503616,None,None,None,2010
1,New Business,2010-06-26,2011-06-26,None,None,906467,Urban,G,10000001,Group Health,...,Finance,55953399,13443077,2,4431744,526910,None,None,None,2010
2,New Business,2010-10-11,2011-10-11,None,None,906468,Urban,G,10000002,Group Health,...,Corporate Services,40825559,22908871,2,1548122,525874,None,None,None,2010
3,New Business,2010-10-03,2011-10-03,None,None,906469,Urban,G,10000003,Group Health,...,Tech,78048354,11453662,2,3313407,464343,None,None,None,2010
4,New Business,2010-04-22,2011-04-22,None,None,906470,Suburban,G,10000004,Family,...,None,None,None,6,1679880,239177,None,None,None,2010


In [18]:
dataset.shape

(1002691, 22)

#### Adding values for columns Deductible or Copay Flag, Policy Deductible, Policy Copay

In [19]:
# Define minimum deductible thresholds by policy type
minimum_deductibles = {
    "I": 1000,  # Individual Plans
    "F": 2500,  # Family Plans
    "HD": 4000,  # High Deductible Plans
    "G": 500,  # Group Plans
    "M": 250,  # Medicare Plans
    "MA": 250,  # Medicare Advantage Plans
    "MCP-HMO": 500,  # Managed Care HMO
    "MCP-PPO": 500,  # Managed Care PPO
    "SNP": 0,  # Special Needs Plans
    "SHI": 500,  # Short-Term Plans
    "CHI": 1000,  # Catastrophic Plans
    "DVI": 250,  # Dental and Vision
}

# Define standardized copay percentages by policy type
standardized_copays = {
    "I": None,  # No copay for Individual Plans
    "F": None,  # No copay for Family Plans
    "HD": None,  # No copay for High Deductible Plans
    "G": 15,  # 15% copay for Group Plans
    "M": 10,  # 10% copay for Medicare Plans
    "MA": 10,  # 10% copay for Medicare Advantage Plans
    "MCP-HMO": 5,  # 5% copay for HMO
    "MCP-PPO": 5,  # 5% copay for PPO
    "SNP": 5,  # 5% copay for Special Needs Plans
    "SHI": 20,  # 20% copay for Short-Term Plans
    "CHI": 15,  # 15% copay for Catastrophic Plans
    "DVI": 20,  # 20% copay for Dental and Vision
}

# Assign Deductible or CoPay Flag
for policy in dataset["Policy Type"].unique():
    mask = dataset["Policy Type"] == policy

    if policy in minimum_deductibles and minimum_deductibles[policy] > 0:
        dataset.loc[mask, "Deductible or CoPay Flag"] = "Deductible"
        dataset.loc[mask, "Deductible Amount"] = minimum_deductibles[policy]
        dataset.loc[mask, "CoPay Percentage"] = None
    elif policy in standardized_copays and standardized_copays[policy] is not None:
        dataset.loc[mask, "Deductible or CoPay Flag"] = "Copay"
        dataset.loc[mask, "CoPay Percentage"] = standardized_copays[policy]
        dataset.loc[mask, "Deductible Amount"] = None

# Apply location-based adjustments to Deductible
location_types = dataset["Location Type"]
dataset.loc[(location_types == "Urban") & (dataset["Deductible Amount"].notna()), "Deductible Amount"] *= 1.1
dataset.loc[(location_types == "Suburban") & (dataset["Deductible Amount"].notna()), "Deductible Amount"] *= 1.05

# Adjust Deductible and Copay for renewals
renewal_as_is_mask = dataset["Renewal Flag"] == "Renewed As-Is"
renewal_upsell_mask = dataset["Renewal Flag"] == "Renewed with Upsell"

# Adjust Deductibles
dataset.loc[renewal_as_is_mask & (dataset["Deductible or CoPay Flag"] == "Deductible"), "Deductible Amount"] *= np.random.uniform(0.97, 0.99, size=renewal_as_is_mask.sum())
dataset.loc[renewal_upsell_mask & (dataset["Deductible or CoPay Flag"] == "Deductible"), "Deductible Amount"] *= np.random.uniform(0.90, 0.95, size=renewal_upsell_mask.sum())

# Adjust Copays
dataset.loc[renewal_upsell_mask & (dataset["Deductible or CoPay Flag"] == "Copay"), "CoPay Percentage"] *= np.random.uniform(0.90, 0.95, size=renewal_upsell_mask.sum())

# Format the Deductible Amount and CoPay Percentage columns
dataset["Deductible Amount"] = dataset["Deductible Amount"].fillna(0).astype(int)
dataset["CoPay Percentage"] = dataset["CoPay Percentage"].fillna(0).astype(float).round(1)

print("Deductible and CoPay values successfully assigned.")

dataset.head()

<ipython-input-19-cf0cf3181902>:63: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataset["Deductible Amount"] = dataset["Deductible Amount"].fillna(0).astype(int)


Deductible and CoPay values successfully assigned.


<ipython-input-19-cf0cf3181902>:64: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  dataset["CoPay Percentage"] = dataset["CoPay Percentage"].fillna(0).astype(float).round(1)


,Business Type,Policy Start Date,Policy End Date,Churn Flag,Renewal Flag,Policyholder ID,Location Type,Policy Type,Policy ID,Group Type,...,Industry Type,Annual Revenue,Annual Payroll,Number of Members,Coverage Amount,Policy Annual Premium,Deductible or CoPay Flag,Deductible Amount,CoPay Percentage,Year
0,New Business,2010-03-15,2011-03-15,None,None,906466,Urban,G,10000000,Family,...,None,None,None,3,1856126,503616,Deductible,550,0.0,2010
1,New Business,2010-06-26,2011-06-26,None,None,906467,Urban,G,10000001,Group Health,...,Finance,55953399,13443077,2,4431744,526910,Deductible,550,0.0,2010
2,New Business,2010-10-11,2011-10-11,None,None,906468,Urban,G,10000002,Group Health,...,Corporate Services,40825559,22908871,2,1548122,525874,Deductible,550,0.0,2010
3,New Business,2010-10-03,2011-10-03,None,None,906469,Urban,G,10000003,Group Health,...,Tech,78048354,11453662,2,3313407,464343,Deductible,550,0.0,2010
4,New Business,2010-04-22,2011-04-22,None,None,906470,Suburban,G,10000004,Family,...,None,None,None,6,1679880,239177,Deductible,525,0.0,2010


In [20]:
dataset.shape

(1002691, 22)

In [21]:
dataset = dataset.drop(columns=["Year"], axis = 1)
dataset.head()

,Business Type,Policy Start Date,Policy End Date,Churn Flag,Renewal Flag,Policyholder ID,Location Type,Policy Type,Policy ID,Group Type,...,Corporation Type,Industry Type,Annual Revenue,Annual Payroll,Number of Members,Coverage Amount,Policy Annual Premium,Deductible or CoPay Flag,Deductible Amount,CoPay Percentage
0,New Business,2010-03-15,2011-03-15,None,None,906466,Urban,G,10000000,Family,...,None,None,None,None,3,1856126,503616,Deductible,550,0.0
1,New Business,2010-06-26,2011-06-26,None,None,906467,Urban,G,10000001,Group Health,...,Large Corporation,Finance,55953399,13443077,2,4431744,526910,Deductible,550,0.0
2,New Business,2010-10-11,2011-10-11,None,None,906468,Urban,G,10000002,Group Health,...,Medium Corporation,Corporate Services,40825559,22908871,2,1548122,525874,Deductible,550,0.0
3,New Business,2010-10-03,2011-10-03,None,None,906469,Urban,G,10000003,Group Health,...,Large Corporation,Tech,78048354,11453662,2,3313407,464343,Deductible,550,0.0
4,New Business,2010-04-22,2011-04-22,None,None,906470,Suburban,G,10000004,Family,...,None,None,None,None,6,1679880,239177,Deductible,525,0.0


#### Writing the Synthetic Policy Dataset to a CSV File and then Downloaded


In [22]:
# Save the dataset to a local file in Colab
file_path = 'inpatient_healthcare_insurance_synthetic_policy_dataset.csv'
dataset.to_csv(file_path, index=False)
print(f"Dataset saved temporarily at {file_path}")

Dataset saved temporarily at inpatient_healthcare_insurance_synthetic_policy_dataset.csv


In [23]:
from google.colab import files

# Download the file
files.download(file_path)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>